# 04 — Hyperparameter Tuning, Calibration & Threshold Analysis

This notebook:
1. Loads processed features and creates 60/20/20 stratified splits
2. Tunes Logistic Regression, Random Forest, and Gradient Boosting with `RandomizedSearchCV`
3. Compares tuned models on the validation set
4. Calibrates the best model (sigmoid vs isotonic)
5. Sweeps decision thresholds and identifies the optimal operating point
6. Saves the best tuned model and a metrics summary

**Dataset:** `hospital_readmission_risk_dataset_2026_v1_18000rows.csv`  
**Target:** `Readmitted_Within_30_Days` (binary, ~74% positive rate)

## 0. Environment Setup

In [1]:
import sys
from pathlib import Path

# Resolve project root (notebooks/ → parent)
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root : {ROOT}")
print(f"Python       : {sys.executable}")

Project root : C:\Users\speak\HA_Project\Hospital-Readmission-Project-
Python       : C:\Users\speak\HA_Project\Hospital-Readmission-Project-\.venv\Scripts\python.exe


In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import load_config, set_seed
from src.modeling import (
    load_features,
    make_splits,
    build_baselines,
    build_param_distributions,
    tune_model,
    evaluate,
    compare_models,
    select_best_model,
    plot_roc_curves,
    plot_pr_curves,
    plot_confusion_matrix,
    plot_feature_importance,
    calibrate_model,
    threshold_sweep,
    plot_threshold_analysis,
)

# Load config and inject base_dir for path resolution
config = load_config(ROOT / "config" / "config.yaml")
config["_base_dir"] = str(ROOT)

SEED = config.get("random_seed", 42)
set_seed(SEED)

print("Config loaded. Random seed:", SEED)

2026-03-09 23:37:49 | INFO     | src.modeling | XGBoost not installed — using HistGradientBoostingClassifier.


Config loaded. Random seed: 42


## 1. Load Features & Create Splits

In [3]:
X, y, feature_names = load_features(config, base_dir=ROOT)

print(f"Feature matrix : {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"Target positives: {y.mean():.1%}")
print(f"\nFirst 5 features: {feature_names[:5]}")

2026-03-09 23:37:49 | INFO     | src.modeling | Loaded feature dataset: 18000 rows × 45 columns


2026-03-09 23:37:49 | INFO     | src.modeling | Sanitised 1 column name(s): [('Discharge_Disposition_Nursing Facility', 'Discharge_Disposition_Nursing_Facility')]


2026-03-09 23:37:49 | INFO     | src.modeling | Excluded leakage column(s): ['Followup_Appointment_Scheduled', 'Medication_Adherence_Score']


2026-03-09 23:37:49 | INFO     | src.modeling | Dropped redundant column(s): ['age_group_<40', 'age_group_40-64', 'age_group_65-79', 'age_group_80+', 'discharge_disposition_cat_Facility', 'discharge_disposition_cat_Home']


2026-03-09 23:37:49 | INFO     | src.modeling | Feature matrix: 18000 rows × 36 features | pos=13349 (74.2%) neg=4651 (25.8%)


Feature matrix : 18,000 rows × 36 features
Target positives: 74.2%

First 5 features: ['Age', 'Socioeconomic_Risk_Score', 'Previous_Admissions_6M', 'Previous_Readmissions_1Y', 'Time_Since_Last_Discharge']


In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y, config)

print(f"Train : {len(y_train):,} rows  ({y_train.mean():.1%} positive)")
print(f"Val   : {len(y_val):,} rows  ({y_val.mean():.1%} positive)")
print(f"Test  : {len(y_test):,} rows  ({y_test.mean():.1%} positive)")

2026-03-09 23:37:49 | INFO     | src.modeling | Split: train=10800 | val=3600 | test=3600


Train : 10,800 rows  (74.2% positive)
Val   : 3,600 rows  (74.2% positive)
Test  : 3,600 rows  (74.2% positive)


## 2. Feature Overview

In [5]:
# Categorise features
numeric_feats = X_train.select_dtypes(include="number").columns.tolist()
bool_feats    = X_train.select_dtypes(include="bool").columns.tolist()
other_feats   = [c for c in feature_names if c not in numeric_feats + bool_feats]

print(f"Numeric features  : {len(numeric_feats)}")
print(f"Boolean features  : {len(bool_feats)}")
print(f"Other features    : {len(other_feats)}")

# Show interaction features that were added
interaction_feats = [c for c in feature_names if "_x_" in c]
print(f"\nInteraction features ({len(interaction_feats)}):")
for f in interaction_feats:
    print(f"  {f}")

Numeric features  : 36
Boolean features  : 0
Other features    : 0

Interaction features (2):
  Age_x_Comorbidity_Index
  Severity_Score_x_Length_of_Stay


## 3. Baseline Reference

Quick baseline evaluation for comparison against tuned models.

In [6]:
baselines = build_baselines(config)

baseline_results = {}
for name, model in baselines.items():
    model.fit(X_train, y_train)
    baseline_results[name] = evaluate(model, X_val, y_val)
    baseline_results[name]["model_name"] = name
    print(f"{name:<25} ROC-AUC={baseline_results[name]['roc_auc']:.4f}  "
          f"PR-AUC={baseline_results[name]['pr_auc']:.4f}  "
          f"F1={baseline_results[name]['f1']:.4f}")

2026-03-09 23:37:49 | INFO     | src.modeling | Baseline estimators: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


LogisticRegression        ROC-AUC=0.5667  PR-AUC=0.7788  F1=0.6488


RandomForest              ROC-AUC=0.5483  PR-AUC=0.7692  F1=0.8232


GradientBoosting          ROC-AUC=0.5530  PR-AUC=0.7655  F1=0.6522


## 4. Hyperparameter Tuning

`RandomizedSearchCV` with `StratifiedKFold` (5 folds, 20 iterations each).  
Scoring: **ROC-AUC** (chosen for cleaner signal on this imbalanced dataset).

> **Runtime note:** Each model runs 100 fits (20 iter × 5 folds). Expect ~2–5 minutes total.

### 4a. Logistic Regression

In [7]:
lr_baseline = build_baselines(config)["LogisticRegression"]
lr_param_dist = build_param_distributions("LogisticRegression", config)

print("LR search space:")
for k, v in lr_param_dist.items():
    print(f"  {k}: {v}")

lr_result = tune_model(lr_baseline, lr_param_dist, X_train, y_train, config)

print(f"\nBest CV ROC-AUC : {lr_result['cv_score']:.4f}")
print(f"Best params     : {lr_result['best_params']}")

2026-03-09 23:37:51 | INFO     | src.modeling | Baseline estimators: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


2026-03-09 23:37:51 | INFO     | src.modeling | RandomizedSearchCV: model=Pipeline  n_iter=20  cv=5  scoring=roc_auc


LR search space:
  classifier__C: [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0]
  classifier__penalty: ['l1', 'l2']
  classifier__solver: ['liblinear', 'saga']


2026-03-09 23:38:00 | INFO     | src.modeling | Best roc_auc: 0.5801 | params: {'classifier__solver': 'liblinear', 'classifier__penalty': 'l1', 'classifier__C': 0.1}



Best CV ROC-AUC : 0.5801
Best params     : {'classifier__solver': 'liblinear', 'classifier__penalty': 'l1', 'classifier__C': 0.1}


### 4b. Random Forest

In [8]:
rf_baseline = build_baselines(config)["RandomForest"]
rf_param_dist = build_param_distributions("RandomForest", config)

print("RF search space:")
for k, v in rf_param_dist.items():
    print(f"  {k}: {v}")

rf_result = tune_model(rf_baseline, rf_param_dist, X_train, y_train, config)

print(f"\nBest CV ROC-AUC : {rf_result['cv_score']:.4f}")
print(f"Best params     : {rf_result['best_params']}")

2026-03-09 23:38:00 | INFO     | src.modeling | Baseline estimators: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


2026-03-09 23:38:00 | INFO     | src.modeling | RandomizedSearchCV: model=RandomForestClassifier  n_iter=20  cv=5  scoring=roc_auc


RF search space:
  n_estimators: [100, 150, 200, 300]
  max_depth: [5, 10, 15, 20, None]
  min_samples_split: [2, 5, 10]
  min_samples_leaf: [5, 10, 20]


2026-03-09 23:38:31 | INFO     | src.modeling | Best roc_auc: 0.5785 | params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 5}



Best CV ROC-AUC : 0.5785
Best params     : {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 10, 'max_depth': 5}


### 4c. Gradient Boosting

In [9]:
gb_baseline = build_baselines(config)["GradientBoosting"]
gb_param_dist = build_param_distributions("GradientBoosting", config)

print("GB search space:")
for k, v in gb_param_dist.items():
    print(f"  {k}: {v}")

gb_result = tune_model(gb_baseline, gb_param_dist, X_train, y_train, config)

print(f"\nBest CV ROC-AUC : {gb_result['cv_score']:.4f}")
print(f"Best params     : {gb_result['best_params']}")

2026-03-09 23:38:31 | INFO     | src.modeling | Baseline estimators: ['LogisticRegression', 'RandomForest', 'GradientBoosting']


2026-03-09 23:38:31 | INFO     | src.modeling | RandomizedSearchCV: model=HistGradientBoostingClassifier  n_iter=20  cv=5  scoring=roc_auc


GB search space:
  learning_rate: [0.01, 0.05, 0.1, 0.2]
  max_iter: [100, 150, 200, 300]
  max_depth: [3, 4, 5, 6, 7]
  min_samples_leaf: [10, 20, 30, 50]


2026-03-09 23:38:50 | INFO     | src.modeling | Best roc_auc: 0.5731 | params: {'min_samples_leaf': 50, 'max_iter': 150, 'max_depth': 6, 'learning_rate': 0.01}



Best CV ROC-AUC : 0.5731
Best params     : {'min_samples_leaf': 50, 'max_iter': 150, 'max_depth': 6, 'learning_rate': 0.01}


## 5. Compare Tuned Models

In [10]:
tuned_models = {
    "LogisticRegression": lr_result["estimator"],
    "RandomForest":       rf_result["estimator"],
    "GradientBoosting":   gb_result["estimator"],
}

tuned_results = {}
for name, model in tuned_models.items():
    tuned_results[name] = evaluate(model, X_val, y_val)

summary_df = compare_models(tuned_results)

# Add baseline reference rows
baseline_rows = []
for name, res in baseline_results.items():
    row = {"model_name": name + " (baseline)",
           "roc_auc":    res["roc_auc"],
           "pr_auc":     res["pr_auc"],
           "f1":         res["f1"],
           "recall":     res["recall"],
           "precision":  res["precision"]}
    baseline_rows.append(row)

full_summary = pd.concat([summary_df,
                           pd.DataFrame(baseline_rows).set_index("model_name")],
                          axis=0)

display(full_summary.style.highlight_max(
    subset=["roc_auc", "pr_auc", "f1", "recall"],
    color="lightgreen"
).format("{:.4f}"))

,roc_auc,pr_auc,accuracy,precision,recall,f1,specificity,brier
LogisticRegression,0.5668,0.7795,0.5503,0.7773,0.5517,0.6453,0.5462,0.2465
GradientBoosting,0.5544,0.7656,0.5444,0.7760,0.5423,0.6384,0.5505,0.2441
RandomForest,0.5526,0.7728,0.5567,0.7677,0.5768,0.6587,0.4989,0.2445
LogisticRegression (baseline),0.5667,0.7788,nan,0.7755,0.5577,0.6488,nan,nan
RandomForest (baseline),0.5483,0.7692,nan,0.7484,0.9146,0.8232,nan,nan
GradientBoosting (baseline),0.5530,0.7655,nan,0.7731,0.5640,0.6522,nan,nan


In [11]:
# Gain over baseline
print("=== Tuning Gains (val ROC-AUC) ===")
for name in ["LogisticRegression", "RandomForest", "GradientBoosting"]:
    base_auc  = baseline_results[name]["roc_auc"]
    tuned_auc = tuned_results[name]["roc_auc"]
    delta     = tuned_auc - base_auc
    print(f"  {name:<25} {base_auc:.4f} → {tuned_auc:.4f}  Δ={delta:+.4f}")

=== Tuning Gains (val ROC-AUC) ===
  LogisticRegression        0.5667 → 0.5668  Δ=+0.0001
  RandomForest              0.5483 → 0.5526  Δ=+0.0043
  GradientBoosting          0.5530 → 0.5544  Δ=+0.0014


## 6. ROC & PR Curves — Tuned Models

In [12]:
plot_roc_curves(
    tuned_models, X_val, y_val, config,
    title="ROC Curves — Tuned Models (Validation)",
    filename="roc_curve_tuned_models.png"
)

2026-03-09 23:38:50 | INFO     | src.modeling | Saved ROC curve → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\roc_curve_tuned_models.png


In [13]:
NO_SKILL_PRAUC = float(y_val.mean())

plot_pr_curves(
    tuned_models, X_val, y_val, config,
    title=f"PR Curves — Tuned Models (Validation)  [no-skill={NO_SKILL_PRAUC:.3f}]",
    filename="pr_curve_tuned_models.png"
)

2026-03-09 23:38:51 | INFO     | src.modeling | Saved PR curve → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\pr_curve_tuned_models.png


## 7. Confusion Matrices — Tuned Models

In [14]:
for name, model in tuned_models.items():
    plot_confusion_matrix(
        model, X_val, y_val, name, config,
        threshold=0.5,
        filename=f"confusion_matrix_tuned_{name.lower()}.png"
    )

2026-03-09 23:38:51 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\confusion_matrix_tuned_logisticregression.png


2026-03-09 23:38:51 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\confusion_matrix_tuned_randomforest.png


2026-03-09 23:38:51 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\confusion_matrix_tuned_gradientboosting.png


## 8. Select Best Tuned Model

In [15]:
best_name = select_best_model(summary_df, metric="roc_auc")
best_model = tuned_models[best_name]

print(f"Best tuned model : {best_name}")
print(f"Val ROC-AUC      : {tuned_results[best_name]['roc_auc']:.4f}")
print(f"Val PR-AUC       : {tuned_results[best_name]['pr_auc']:.4f}")
print(f"Val F1           : {tuned_results[best_name]['f1']:.4f}")
print(f"Val Recall       : {tuned_results[best_name]['recall']:.4f}")

2026-03-09 23:38:51 | INFO     | src.modeling | Best model by roc_auc: LogisticRegression (0.5668)


Best tuned model : LogisticRegression
Val ROC-AUC      : 0.5668
Val PR-AUC       : 0.7795
Val F1           : 0.6453
Val Recall       : 0.5517


## 9. Probability Calibration

Compare **sigmoid** (Platt scaling) vs **isotonic** regression calibration.  
The method with the lower Brier score on the calibration (validation) set is kept.

In [16]:
calibrated_model = calibrate_model(
    best_model, X_val, y_val, config
)

# Evaluate calibrated model on validation set
cal_metrics = evaluate(calibrated_model, X_val, y_val)
raw_metrics = tuned_results[best_name]

print("\n=== Calibration Impact ===")
print(f"{'Metric':<12} {'Pre-cal':>10} {'Post-cal':>10}")
print("-" * 35)
for m in ["roc_auc", "pr_auc", "brier", "f1", "recall"]:
    pre  = raw_metrics.get(m, float("nan"))
    post = cal_metrics.get(m, float("nan"))
    print(f"{m:<12} {pre:>10.4f} {post:>10.4f}")

2026-03-09 23:38:51 | INFO     | src.modeling | Calibration (sigmoid): Brier=0.1896


2026-03-09 23:38:51 | INFO     | src.modeling | Calibration (isotonic): Brier=0.1885


2026-03-09 23:38:51 | INFO     | src.modeling | Best calibration method: isotonic


2026-03-09 23:38:51 | INFO     | src.modeling | Saved calibration comparison → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\calibration_curve.png



=== Calibration Impact ===
Metric          Pre-cal   Post-cal
-----------------------------------
roc_auc          0.5668     0.5742
pr_auc           0.7795     0.7811
brier            0.2465     0.1885
f1               0.6453     0.8519
recall           0.5517     1.0000


## 10. Threshold Sweep & Optimal Operating Point

Sweeps thresholds from 0.05 → 0.95 and finds the point that maximises the
configured objective (`optimize_by` in config — default: F1).

In [17]:
sweep_df, optimal_threshold = threshold_sweep(
    calibrated_model, X_val, y_val, config
)

print(f"Optimal threshold : {optimal_threshold:.2f}")
opt_row = sweep_df[sweep_df["threshold"].round(2) == round(optimal_threshold, 2)].iloc[0]
print(f"  Precision : {opt_row['precision']:.4f}")
print(f"  Recall    : {opt_row['recall']:.4f}")
print(f"  F1        : {opt_row['f1']:.4f}")
print(f"  Specificity: {opt_row['specificity']:.4f}")

sweep_df.head(10)

2026-03-09 23:38:51 | INFO     | src.modeling | Threshold sweep: optimal threshold by f1 = 0.050


Optimal threshold : 0.05
  Precision : 0.7421
  Recall    : 1.0000
  F1        : 0.8519
  Specificity: 0.0022


,threshold,precision,recall,f1,specificity,accuracy
0,0.05,0.742079,1.0,0.851946,0.002151,0.742222
1,0.10,0.742079,1.0,0.851946,0.002151,0.742222
2,0.15,0.742079,1.0,0.851946,0.002151,0.742222
3,0.20,0.742079,1.0,0.851946,0.002151,0.742222
4,0.25,0.742079,1.0,0.851946,0.002151,0.742222
5,0.30,0.742079,1.0,0.851946,0.002151,0.742222
6,0.35,0.742079,1.0,0.851946,0.002151,0.742222
7,0.40,0.742079,1.0,0.851946,0.002151,0.742222
8,0.45,0.742079,1.0,0.851946,0.002151,0.742222
9,0.50,0.742079,1.0,0.851946,0.002151,0.742222


In [18]:
plot_threshold_analysis(
    sweep_df, optimal_threshold, config,
    filename="threshold_analysis.png"
)

2026-03-09 23:38:52 | INFO     | src.modeling | Saved threshold analysis → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\threshold_analysis.png


### Evaluate calibrated model at optimal threshold

In [19]:
opt_metrics = evaluate(calibrated_model, X_val, y_val, threshold=optimal_threshold)

print(f"=== Calibrated {best_name} @ threshold={optimal_threshold:.2f} ===")
print(f"  ROC-AUC   : {opt_metrics['roc_auc']:.4f}")
print(f"  PR-AUC    : {opt_metrics['pr_auc']:.4f}")
print(f"  Precision : {opt_metrics['precision']:.4f}")
print(f"  Recall    : {opt_metrics['recall']:.4f}")
print(f"  F1        : {opt_metrics['f1']:.4f}")
print(f"  Specificity: {opt_metrics['specificity']:.4f}")
print(f"  Brier     : {opt_metrics['brier']:.4f}")

=== Calibrated LogisticRegression @ threshold=0.05 ===
  ROC-AUC   : 0.5742
  PR-AUC    : 0.7811
  Precision : 0.7421
  Recall    : 1.0000
  F1        : 0.8519
  Specificity: 0.0022
  Brier     : 0.1885


In [20]:
plot_confusion_matrix(
    calibrated_model, X_val, y_val,
    f"{best_name} (calibrated, t={optimal_threshold:.2f})",
    config,
    threshold=optimal_threshold,
    filename="confusion_matrix_calibrated_optimal.png"
)

2026-03-09 23:38:52 | INFO     | src.modeling | Saved confusion matrix → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\confusion_matrix_calibrated_optimal.png


## 11. Feature Importance — Tuned Models

In [21]:
for name, model in tuned_models.items():
    try:
        plot_feature_importance(
            model, feature_names, name, config, top_n=20
        )
    except Exception as e:
        print(f"{name}: feature importance not available — {e}")

2026-03-09 23:38:52 | INFO     | src.modeling | Saved feature importance → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\feature_importance_logisticregression.png


2026-03-09 23:38:52 | INFO     | src.modeling | Saved feature importance → C:\Users\speak\HA_Project\Hospital-Readmission-Project-\reports\figures\feature_importance_randomforest.png


2026-03-09 23:38:52 | INFO     | src.modeling | No feature importance available for GradientBoosting — skipping.


## 12. Final Test-Set Evaluation

Run **once** — holdout test set is touched only here.

In [22]:
test_metrics = evaluate(calibrated_model, X_test, y_test, threshold=optimal_threshold)

print(f"=== TEST SET — {best_name} (calibrated, t={optimal_threshold:.2f}) ===")
for k, v in test_metrics.items():
    print(f"  {k:<14} {v:.4f}")

=== TEST SET — LogisticRegression (calibrated, t=0.05) ===
  roc_auc        0.5673
  pr_auc         0.7760
  accuracy       0.7422
  precision      0.7421
  recall         1.0000
  f1             0.8519
  specificity    0.0022
  brier          0.1887


## 13. Save Artifacts

In [23]:
import joblib

model_dir = Path(config["_base_dir"]) / config["paths"]["model_dir"]
model_dir.mkdir(parents=True, exist_ok=True)

# Save best tuned + calibrated model
artifact = {
    "name":          best_name,
    "model":         calibrated_model,
    "feature_names": feature_names,
    "threshold":     optimal_threshold,
    "val_metrics":   opt_metrics,
    "test_metrics":  test_metrics,
    "best_params": {
        "LogisticRegression": lr_result["best_params"],
        "RandomForest":       rf_result["best_params"],
        "GradientBoosting":   gb_result["best_params"],
    },
}

model_path = model_dir / "best_tuned_model.pkl"
joblib.dump(artifact, model_path)
print(f"Saved: {model_path}")

# Save tuned metrics summary
metrics_path = (
    Path(config["_base_dir"]) /
    config["paths"].get("metrics_tuned", "data/processed/metrics_tuned_summary.csv")
)
metrics_path.parent.mkdir(parents=True, exist_ok=True)

rows = []
for name, res in tuned_results.items():
    row = {"split": "validation", "model": name, **res}
    row.pop("model_name", None)
    rows.append(row)

# Add test row for best calibrated model
test_row = {"split": "test", "model": best_name + " (calibrated)", **test_metrics}
rows.append(test_row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path}")

display(metrics_df)

Saved: C:\Users\speak\HA_Project\Hospital-Readmission-Project-\models\best_tuned_model.pkl
Saved: C:\Users\speak\HA_Project\Hospital-Readmission-Project-\data\processed\metrics_tuned_summary.csv


,split,model,roc_auc,pr_auc,accuracy,precision,recall,f1,specificity,brier
0,validation,LogisticRegression,0.566837,0.779517,0.550278,0.777309,0.551685,0.645345,0.546237,0.246486
1,validation,RandomForest,0.552621,0.772777,0.556667,0.767697,0.576779,0.658683,0.498925,0.244454
2,validation,GradientBoosting,0.554373,0.765619,0.544444,0.775991,0.542322,0.638448,0.550538,0.244124
3,test,LogisticRegression (calibrated),0.567281,0.776037,0.742222,0.742079,1.000000,0.851946,0.002151,0.188713


## 14. Summary & Next Steps

### Results Summary

| Stage | ROC-AUC | PR-AUC | F1 | Recall |
|---|---|---|---|---|
| Best baseline (val) | — | — | — | — |
| Best tuned (val)    | — | — | — | — |
| Calibrated + optimal threshold (val) | — | — | — | — |
| **Final test set**  | — | — | — | — |

*(Fill in from output above)*

### What Was Done
- **Hyperparameter tuning**: RandomizedSearchCV with StratifiedKFold (5 folds, 20 iterations)
- **Calibration**: Compared sigmoid vs isotonic; best method selected by Brier score
- **Threshold optimisation**: Swept 0.05–0.95 to maximise F1 at optimal operating point
- **Interaction features**: `Age × Comorbidity_Index`, `Severity_Score × Length_of_Stay`
- **Leakage controls**: Excluded post-discharge features; removed redundant OHE columns

### Known Limitations
- Dataset appears synthetic with near-random signal (ROC-AUC near 0.5–0.6 range)
- The 74% positive rate inverts the typical class-imbalance assumption — no-skill PR-AUC is high
- Calibration on the validation set may slightly overestimate calibration quality (use separate calibration set in production)

### Recommended Next Steps
1. **Real dataset**: Replace synthetic data with actual EHR/claims data for meaningful predictive signal
2. **Clinical feature engineering**: ICD-10 code grouping, medication counts, lab trend features
3. **SHAP explanations**: Add model-agnostic SHAP values for clinical interpretability
4. **Fairness audit**: Evaluate disparate impact across gender, insurance type, and age groups
5. **Deployment**: Wrap the saved `best_tuned_model.pkl` in a REST API (FastAPI/Flask) for real-time scoring